In [ ]:
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from pathlib import Path

In [ ]:
CHILDREN_JSONL = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_children.jsonl"
OUTPUT_EMBEDDINGS = "/home/razor/Documents/alzheimers-rag/data/processed/child_embeddings.npy"
OUTPUT_ID_MAP = "/home/razor/Documents/alzheimers-rag/data/processed/child_id_map.jsonl"  # maps row index -> chunk_id + metadata pointer
MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
BATCH_SIZE = 20          # tune based on your GPU/CPU memory
MAX_LENGTH = 512         # MedCPT's actual hard ceiling — matches your chunking decision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

In [ ]:
def load_children(path):
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    return chunks

def build_encoder_input(chunk):
    """
    MedCPT-Article-Encoder expects [title, text] pairs — it was trained this way,
    so don't just dump raw chunk text in. Use doc title + chunk content.
    Adjust field names to match your actual schema.
    """
    title = chunk.get("doc_title", "")
    text = chunk.get("text", "")
    return [title, text]

def embed_batch(batch_inputs):
    with torch.no_grad():
        encoded = tokenizer(
            batch_inputs,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        outputs = model(**encoded)
        # MedCPT uses the [CLS] token embedding (first token of last_hidden_state)
        embeddings = outputs.last_hidden_state[:, 0, :]
        return embeddings.cpu().numpy()

In [ ]:
def mean_pool_embeddings(attention_mask, embeddings):
    """
    If you want to use mean pooling instead of [CLS] token, uncomment this function
    and replace the embedding extraction in embed_batch with this.
    """
    mask = attention_mask.unsqueeze(-1).expand(embeddings.last_hidden_state.size()).float()
    masked_embeddings = embeddings.last_hidden_state * mask

    sum_embeddings = torch.sum(masked_embeddings, dim=1)
    sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask
    
    return mean_embeddings.cpu().numpy()

In [ ]:
chunks = load_children(CHILDREN_JSONL)
print(f"Loaded {len(chunks)} child chunks")

all_embeddings = []
id_map = []

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    batch_inputs = [build_encoder_input(c) for c in batch]

    embeddings = embed_batch(batch_inputs)
    all_embeddings.append(embeddings)

    for c in batch:
        id_map.append({
            "chunk_id": c["chunk_id"],
            "parent_id": c.get("parent_id"),
            "heading_path": c.get("heading_path"),
        })

    if (i // BATCH_SIZE) % 20 == 0:
        print(f"Embedded {i + len(batch)}/{len(chunks)}")

final_embeddings = np.vstack(all_embeddings)
np.save(OUTPUT_EMBEDDINGS, final_embeddings)

with open(OUTPUT_ID_MAP, "w", encoding="utf-8") as f:
    for entry in id_map:
        f.write(json.dumps(entry) + "\n")

print(f"Done. Shape: {final_embeddings.shape}")
assert final_embeddings.shape[0] == len(id_map), "Embedding count mismatch with ID map!"